In [3]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

def scrape_cfb_2024_receiving():
    url = "https://www.sports-reference.com/cfb/years/2024-receiving.html"
    response = requests.get(url)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")

    # Find the correct table
    table = soup.find("table", id="receiving_standard")
    if not table:
        raise ValueError("Could not find table with id='receiving_standard'.")

    # Get headers from first valid data row
    tbody = table.find("tbody")
    first_valid_row = next(row for row in tbody.find_all("tr") if row.find_all("td"))
    columns = [td['data-stat'] for td in first_valid_row.find_all("td")]

    # Extract player data rows
    data = []
    for row in tbody.find_all("tr"):
        if row.get("class") and "thead" in row.get("class"):
            continue
        tds = row.find_all("td")
        if not tds:
            continue
        values = [td.get_text(strip=True) for td in tds]
        data.append(values)

    # Build DataFrame
    df = pd.DataFrame(data, columns=columns)

    # Convert numeric columns safely
    for col in df.columns:
        if df[col].dtype == object:
            df[col] = pd.to_numeric(df[col].str.replace(',', ''), errors='ignore')

    return df

if __name__ == "__main__":
    df = scrape_cfb_2024_receiving()
    print(df.head())
    df.to_csv("cfb_2024_receiving_standard.csv", index=False)


         name_display  team_name_abbr conf_abbr  games    rec  rec_yds  \
0  Harold Fannin Jr.*   Bowling Green       MAC   13.0  117.0   1555.0   
1          Nick Nash*  San Jose State       MWC   12.0  104.0   1382.0   
2   Tetairoa McMillan         Arizona    Big 12   12.0   84.0   1319.0   
3     Jeremiah Smith*      Ohio State   Big Ten   16.0   76.0   1315.0   
4      Travis Hunter*        Colorado    Big 12   13.0   96.0   1258.0   

   rec_yds_per_rec  rec_td  rec_yds_per_g  rush_att  rush_yds  \
0             13.3    10.0          119.6       9.0      65.0   
1             13.3    16.0          115.2       1.0      -9.0   
2             15.7     8.0          109.9       0.0       0.0   
3             17.3    15.0           82.2       6.0      47.0   
4             13.1    15.0           96.8       2.0       5.0   

   rush_yds_per_att  rush_td  rush_yds_per_g  scrim_att  yds_from_scrimmage  \
0               7.2      1.0             5.0      126.0              1620.0   
1     

In [5]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

def scrape_receiving_year(year):
    url = f"https://www.sports-reference.com/cfb/years/{year}-receiving.html"
    print(f"Scraping {year}...")
    try:
        response = requests.get(url)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")

        table = soup.find("table", id="receiving_standard")
        if not table:
            print(f"[!] Table not found for {year}. Skipping.")
            return

        tbody = table.find("tbody")
        first_valid_row = next(row for row in tbody.find_all("tr") if row.find_all("td"))
        columns = [td['data-stat'] for td in first_valid_row.find_all("td")]

        data = []
        for row in tbody.find_all("tr"):
            if row.get("class") and "thead" in row.get("class"):
                continue
            tds = row.find_all("td")
            if not tds:
                continue
            values = [td.get_text(strip=True) for td in tds]
            data.append(values)

        df = pd.DataFrame(data, columns=columns)

        for col in df.columns:
            if df[col].dtype == object:
                df[col] = pd.to_numeric(df[col].str.replace(',', ''), errors='ignore')

        df.to_csv(f"cfb_{year}_receiving.csv", index=False)
        print(f"✅ Saved: cfb_{year}_receiving.csv")

        time.sleep(1.5)  # be polite to the server
    except Exception as e:
        print(f"[!] Error for {year}: {e}")

if __name__ == "__main__":
    for year in range(2024, 2011, -1):  # From 2024 to 2012 inclusive
        scrape_receiving_year(year)


Scraping 2024...
✅ Saved: cfb_2024_receiving.csv
Scraping 2023...
✅ Saved: cfb_2023_receiving.csv
Scraping 2022...
✅ Saved: cfb_2022_receiving.csv
Scraping 2021...
✅ Saved: cfb_2021_receiving.csv
Scraping 2020...
✅ Saved: cfb_2020_receiving.csv
Scraping 2019...
✅ Saved: cfb_2019_receiving.csv
Scraping 2018...
✅ Saved: cfb_2018_receiving.csv
Scraping 2017...
✅ Saved: cfb_2017_receiving.csv
Scraping 2016...
✅ Saved: cfb_2016_receiving.csv
Scraping 2015...
✅ Saved: cfb_2015_receiving.csv
Scraping 2014...
✅ Saved: cfb_2014_receiving.csv
Scraping 2013...
✅ Saved: cfb_2013_receiving.csv
Scraping 2012...
✅ Saved: cfb_2012_receiving.csv
